# Key steering

Layer 22 seems to be the most interpretable. Let's steer it!

- Train linear regression
- Switch up activations
- Regenerate

Let's work backwards. Does regeneration work?

In [1]:
import torch
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

In [30]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import scipy.io.wavfile as wavfile

# Load model
processor = AutoProcessor.from_pretrained("facebook/musicgen-large")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-large").to("cuda")
model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 14.20it/s]


MusicgenForConditionalGeneration(
  (text_encoder): T5EncoderModel(
    (shared): Embedding(32128, 768)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 768)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
                (relative_attention_bias): Embedding(32, 12)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=768, out_features=3072, bias=False)
                (wo): L

In [66]:
METADATA_PATH = "/home/harinit9/orcd/pool/musicgen-data-nokey/dataset_metadata.json"

with open(METADATA_PATH) as fin:
    metadata = json.load(fin)

LAYER_IDX = 22

In [64]:
# Train logistic regression on layer 22
import sys
sys.path.append("../linear_probe")
from linear_probe_key import train_linear_probe

In [68]:
reg = list(train_linear_probe(22, 128, "max", train=False))
print(f"{reg=}")

[2025-12-09 13:28:51] Logistic regression on layer_idx=22
[2025-12-09 13:28:51] Pooling...
[2025-12-09 13:28:52] Normalizing...
[2025-12-09 13:28:52] Running pca...
[2025-12-09 13:28:53] Making test splits...


reg=[]


In [ ]:
# Load your saved activations - shape (256, 2048)
saved_activations = torch.load(act_path)

# Assuming this is the layer 22 output; add batch dim if needed
layer_acts = saved_activations["decoder.model.decoder.layers.22"]  # adjust key as needed
layer_acts = torch.cat(layer_acts, dim=1)

# Shape should be (batch=1, seq_len=256, hidden=2048)
if layer_acts.dim() == 2:
    layer_acts = layer_acts.unsqueeze(0)
layer_acts = layer_acts.to("cuda")

print(layer_acts.shape)

torch.Size([2, 256, 2048])


In [ ]:
# Track generation step
step_counter = [0]

def patch_hook(module, inp, out):
    """Replace layer 22 output with saved activations at each step."""
    step = step_counter[0]
    assert step < layer_acts.shape[1]

    # During autoregressive generation, we patch one token at a time
    # out[0] is the hidden state, shape (batch, 1, hidden) for each new token
    patched = layer_acts[:, step:step+1, :]
    step_counter[0] += 1
    return (patched,)

# Find and hook layer 22
print("Finding and hooking layer...")

for name, module in model.decoder.named_modules():
    if name == "model.decoder.layers.22":
        module._forward_hooks.clear()
        handle = module.register_forward_hook(patch_hook)
        break

inputs = processor(
    text=clip["prompt"],
    padding=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    print("Generating...")
    audio_values = model.generate(
        **inputs,
        do_sample=False,  # deterministic since we're patching
        max_new_tokens=256,
    )

handle.remove()

# Save audio (5s)
clip_np = audio_values[0, 0].cpu().numpy()
wavfile.write(f"{clip_idx:02d}_reconstructed.wav", rate=32000, data=clip_np)

Finding and hooking layer...
Generating...
